In [1]:
import duckdb
import pandas as pd
import numpy as np
from pathlib import Path

PROJECT_DIR = Path(r"D:\IIT\Year 4\IRP\IRP_Project_Folder")

DATA_RAW = PROJECT_DIR / "data" / "raw"

csv_path = DATA_RAW / "transactions_train.csv"
articles_path = DATA_RAW / "articles.csv"

INTERIM_DIR = PROJECT_DIR / "data" / "interim"
INTERIM_DIR.mkdir(parents=True, exist_ok=True)

con = duckdb.connect()

print("Environment ready!")
print("Transactions found:", csv_path.exists())
print("Articles found:", articles_path.exists())

Environment ready!
Transactions found: True
Articles found: True


In [2]:
weekly_period = con.execute("""
    WITH date_range AS (
        SELECT
            MIN(CAST(t_dat AS DATE)) AS first_date,
            MAX(CAST(t_dat AS DATE)) AS last_date
        FROM read_csv_auto(?)
    ),

    calendar AS (
        SELECT
            CAST(week_start AS DATE) AS week_start
        FROM date_range,
        LATERAL generate_series(
            CAST(date_trunc('week', first_date) AS TIMESTAMP),
            CAST(date_trunc('week', last_date) AS TIMESTAMP),
            INTERVAL 7 DAY
        ) AS weeks(week_start)
    ),

    ranked_weeks AS (
        SELECT
            week_start,
            ROW_NUMBER() OVER (
                ORDER BY week_start
            ) AS week_number,
            COUNT(*) OVER () AS total_weeks
        FROM calendar
    )

    SELECT
        week_start,
        week_number,
        total_weeks
    FROM ranked_weeks
    WHERE week_number = CEIL(total_weeks * 0.70)

""", [str(csv_path)]).df()

training_cutoff = weekly_period.loc[0, "week_start"]
total_weeks = int(weekly_period.loc[0, "total_weeks"])
training_weeks = int(weekly_period.loc[0, "week_number"])

print("Total calendar weeks:", total_weeks)
print("Training weeks:", training_weeks)
print("Training cutoff:", training_cutoff)

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

Total calendar weeks: 106
Training weeks: 75
Training cutoff: 2020-02-17 00:00:00


In [3]:
article_metrics = con.execute("""
    WITH weekly_demand AS (

        SELECT
            article_id,

            CAST(
                date_trunc('week', CAST(t_dat AS DATE))
                AS DATE
            ) AS week_start,

            COUNT(*) AS weekly_demand

        FROM read_csv_auto(?)

        WHERE CAST(
            date_trunc('week', CAST(t_dat AS DATE))
            AS DATE
        ) <= CAST(? AS DATE)

        GROUP BY article_id, week_start
    ),

    article_bounds AS (

        SELECT
            article_id,
            MIN(week_start) AS first_week,
            MAX(week_start) AS last_week,
            SUM(weekly_demand) AS total_units

        FROM weekly_demand
        GROUP BY article_id
    ),

    article_week_grid AS (

        SELECT
            b.article_id,
            b.total_units,
            CAST(w.week_start AS DATE) AS week_start

        FROM article_bounds b

        CROSS JOIN LATERAL generate_series(
            CAST(b.first_week AS TIMESTAMP),
            CAST(b.last_week AS TIMESTAMP),
            INTERVAL 7 DAY
        ) AS w(week_start)
    ),

    complete_weekly_demand AS (

        SELECT
            g.article_id,
            g.total_units,
            COALESCE(d.weekly_demand, 0) AS weekly_demand

        FROM article_week_grid g

        LEFT JOIN weekly_demand d
            ON g.article_id = d.article_id
            AND g.week_start = d.week_start
    ),

    demand_statistics AS (

        SELECT
            article_id,

            MAX(total_units) AS total_units,

            COUNT(*) AS observed_weeks,

            COUNT(*) FILTER (
                WHERE weekly_demand > 0
            ) AS nonzero_demand_weeks,

            AVG(weekly_demand) FILTER (
                WHERE weekly_demand > 0
            ) AS mean_nonzero_demand,

            STDDEV_POP(weekly_demand) FILTER (
                WHERE weekly_demand > 0
            ) AS std_nonzero_demand

        FROM complete_weekly_demand

        GROUP BY article_id
    )

    SELECT
        article_id,
        total_units,
        observed_weeks,
        nonzero_demand_weeks,

        observed_weeks * 1.0
            / NULLIF(nonzero_demand_weeks, 0) AS ADI,

        POWER(
            COALESCE(std_nonzero_demand, 0)
            / NULLIF(mean_nonzero_demand, 0),
            2
        ) AS CV2

    FROM demand_statistics

""", [str(csv_path), training_cutoff]).df()

print("Demand characteristics calculated!")
print("Number of articles:", len(article_metrics))

display(article_metrics.head())

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

Demand characteristics calculated!
Number of articles: 84378


,article_id,total_units,observed_weeks,nonzero_demand_weeks,ADI,CV2
0,0723371002,298.0,45,34,1.323529,0.661637
1,0700664001,44.0,48,26,1.846154,0.557851
2,0794825001,75.0,36,30,1.200000,0.328000
3,0767577009,144.0,25,20,1.250000,1.824074
4,0693246005,2113.0,57,34,1.676471,2.518462


In [4]:
def classify_demand(row):

    adi = row["ADI"]
    cv2 = row["CV2"]

    # Articles with insufficient demand observations
    if row["nonzero_demand_weeks"] < 2:
        return "insufficient_history"

    if pd.isna(adi) or pd.isna(cv2):
        return "insufficient_history"

    if adi <= 1.32 and cv2 <= 0.49:
        return "smooth"

    elif adi <= 1.32 and cv2 > 0.49:
        return "erratic"

    elif adi > 1.32 and cv2 <= 0.49:
        return "intermittent"

    else:
        return "lumpy"


article_metrics["demand_pattern"] = article_metrics.apply(
    classify_demand,
    axis=1
)

display(
    article_metrics["demand_pattern"]
    .value_counts()
    .rename_axis("Demand pattern")
    .reset_index(name="Number of articles")
)

,Demand pattern,Number of articles
0,intermittent,24294
1,lumpy,23930
2,erratic,19955
3,smooth,11228
4,insufficient_history,4971


In [5]:
# Group 4: Intermittent and lumpy articles
intermittent_lumpy = article_metrics[
    article_metrics["demand_pattern"].isin(
        ["intermittent", "lumpy"]
    )
].copy()

# Remove intermittent/lumpy articles from the remaining population
remaining_articles = article_metrics[
    ~article_metrics["demand_pattern"].isin(
        ["intermittent", "lumpy"]
    )
].copy()

# Rank the remaining articles by total training-period demand
remaining_articles["volume_rank"] = (
    remaining_articles["total_units"]
    .rank(method="first", ascending=True)
)

# Divide remaining articles into three approximately equal groups
remaining_articles["volume_group"] = pd.qcut(
    remaining_articles["volume_rank"],
    q=3,
    labels=["Low-volume", "Medium-volume", "High-volume"]
)

# Display group sizes
print("Intermittent/lumpy articles:", len(intermittent_lumpy))

print("\nVolume group distribution:")
display(
    remaining_articles["volume_group"]
    .value_counts()
    .sort_index()
)

Intermittent/lumpy articles: 48224

Volume group distribution:


volume_group
Low-volume       12052
Medium-volume    12051
High-volume      12051
Name: count, dtype: int64

In [6]:
SAMPLE_SIZE = 125
RANDOM_SEED = 42

# Select up to 125 intermittent/lumpy articles
selected_intermittent = intermittent_lumpy.sample(
    n=min(SAMPLE_SIZE, len(intermittent_lumpy)),
    random_state=RANDOM_SEED
).copy()

selected_intermittent["selection_group"] = "Intermittent/Lumpy"

# Select up to 125 articles from each volume group
selected_volume_groups = []

for group in ["High-volume", "Medium-volume", "Low-volume"]:

    candidates = remaining_articles[
        remaining_articles["volume_group"] == group
    ]

    selected = candidates.sample(
        n=min(SAMPLE_SIZE, len(candidates)),
        random_state=RANDOM_SEED
    ).copy()

    selected["selection_group"] = group

    selected_volume_groups.append(selected)

# Combine all selected articles
top500_df = pd.concat(
    [selected_intermittent] + selected_volume_groups,
    ignore_index=True
)

print("Total selected articles:", len(top500_df))
print("Unique article IDs:", top500_df["article_id"].nunique())

display(
    top500_df["selection_group"]
    .value_counts()
    .rename_axis("Selection group")
    .reset_index(name="Selected articles")
)

Total selected articles: 500
Unique article IDs: 500


,Selection group,Selected articles
0,Intermittent/Lumpy,125
1,High-volume,125
2,Medium-volume,125
3,Low-volume,125


In [7]:
print("SELECTION VALIDATION")

print("Total selected:", len(top500_df))

print(
    "Unique article IDs:",
    top500_df["article_id"].nunique()
)

print(
    "Missing article IDs:",
    top500_df["article_id"].isna().sum()
)

print(
    "Duplicate article IDs:",
    top500_df["article_id"].duplicated().sum()
)

print("\nGROUP COUNTS:")

display(
    top500_df.groupby("selection_group")
    .agg(
        selected_articles=("article_id", "count"),
        total_training_demand=("total_units", "sum"),
        average_ADI=("ADI", "mean"),
        average_CV2=("CV2", "mean")
    )
)

SELECTION VALIDATION
Total selected: 500
Unique article IDs: 500
Missing article IDs: 0
Duplicate article IDs: 0

GROUP COUNTS:


,selected_articles,total_training_demand,average_ADI,average_CV2
selection_group,,,,
High-volume,125,136365.0,1.083359,1.371224
Intermittent/Lumpy,125,15165.0,2.606956,0.793109
Low-volume,125,1756.0,1.057333,0.202384
Medium-volume,125,19584.0,1.129281,0.842792


In [8]:
output_path = (
    INTERIM_DIR / "stratified_500_articles.csv"
)

top500_df.to_csv(output_path, index=False)

print("Stratified article selection saved!")
print("File location:", output_path)

Stratified article selection saved!
File location: D:\IIT\Year 4\IRP\IRP_Project_Folder\data\interim\stratified_500_articles.csv


In [9]:
print("Selected articles:", len(top500_df))
print("Unique IDs:", top500_df["article_id"].nunique())

print(
    "Selection file exists:",
    (INTERIM_DIR / "stratified_500_articles.csv").exists()
)

Selected articles: 500
Unique IDs: 500
Selection file exists: True
